In [2]:
import pandas as pd
import numpy as np
import openpyxl

In [3]:
df = pd.read_excel("C:/Users/ander/OneDrive/Desktop/Reto_07_morado/Datos/Originales/información_préstamos.xlsx")

df = df.dropna(subset=["Prima"]).copy()

df.head()

,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,Estudios,Tipo_Jornada_Laboral,Estado_Civil,Posesion_Hipoteca,Personas_Cargo,Proposito,Fiador,Impago,Prima
0,S97R7X,18,61628,83011,397,113,1,8.06,48,0.45,Doctorado,Autónomo,Casado,1,0,Automóvil,0,0,155.80
1,T3ZE0N,69,19485,25474,784,46,2,15.04,48,0.15,Grado Universitario,Tiempo parcial,Soltero,0,1,Educación,0,0,24.20
2,RLGTBY,50,82410,68642,486,14,3,21.96,12,0.71,Escolar,Tiempo parcial,Divorciado,1,0,Automóvil,1,1,58.33
3,BZ86CV,64,132974,208339,308,10,1,24.26,12,0.61,Máster,Desempleado,Casado,1,0,Negocios,1,0,284.46
4,5OD75M,62,51411,113847,412,47,2,5.73,36,0.70,Grado Universitario,Autónomo,Casado,0,0,Negocios,0,0,101.77


In [4]:
df = df[df["Proposito"] == "Vivienda"]

In [5]:
df.duplicated(subset="ID").sum()

#duplicados

np.int64(0)

In [6]:
df.isna().sum()

#nulos

ID                      0
Edad                    0
Ingresos                0
Monto_Inicial           0
Scoring_Crediticio      0
Meses_Empleo            0
Num_Creditos            0
Ratio_Interes           0
Duracion                0
Ratio_Deuda_Ingresos    0
Estudios                0
Tipo_Jornada_Laboral    0
Estado_Civil            0
Posesion_Hipoteca       0
Personas_Cargo          0
Proposito               0
Fiador                  0
Impago                  0
Prima                   0
dtype: int64

In [7]:
df["Meses_Maximos"] = (df["Edad"] - 16) * 12
df_invalidos = df[df["Meses_Empleo"] > df["Meses_Maximos"]]

In [8]:
df = df[df["Meses_Empleo"] <= df["Meses_Maximos"]]
df = df.drop(columns="Meses_Maximos")

## Se eliminaron registros con cosas no lógicas en la variable Meses_Empleo, ya que implicaban experiencia laboral previa 
# a la edad legal mínima (16 años).

In [9]:
df = df[(df["Edad"] >= 16) & (df["Edad"] <= 100)]

# edades, no menor a 16 y mayores a 100

In [10]:
df["Tipo_Jornada_Laboral"].value_counts()

Tipo_Jornada_Laboral
Autónomo            12011
Desempleado         11975
Tiempo parcial      11972
Jornada completa    11902
Name: count, dtype: int64

In [11]:
df["Tipo_Jornada_Laboral"] = (
    df["Tipo_Jornada_Laboral"]
    .str.strip()
    .str.lower()
)

df["Tipo_Jornada_Laboral"] = df["Tipo_Jornada_Laboral"].replace({
    "autonomo": "autónomo"
})

df["Tipo_Jornada_Laboral"].value_counts()

## por si hay alguna palabra mal, en plan espacios masculino/femenino

Tipo_Jornada_Laboral
autónomo            12011
desempleado         11975
tiempo parcial      11972
jornada completa    11902
Name: count, dtype: int64

In [12]:
df[~df["Fiador"].isin([0,1])]

## tiene que ser binario

,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,Estudios,Tipo_Jornada_Laboral,Estado_Civil,Posesion_Hipoteca,Personas_Cargo,Proposito,Fiador,Impago,Prima


In [13]:
df = df[df["Ingresos"] > 0]

## ingresos siempre positivos, no se pueden negativos

In [14]:
df = df[df["Monto_Inicial"] > 0]

## monto inicial imposible negativo

In [15]:
df = df[(df["Ratio_Deuda_Ingresos"] >= 0) & (df["Ratio_Deuda_Ingresos"] <= 1)]

# no puedes menos del 0 y no puedes deber mas de 1

In [16]:
df = df[(df["Ratio_Interes"] > 0) & (df["Ratio_Interes"] <= 100)]

# no se puede menos de 0 ni mas de 100

In [17]:
df.describe()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 47860 entries, 6 to 255345
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ID                    47860 non-null  object 
 1   Edad                  47860 non-null  int64  
 2   Ingresos              47860 non-null  int64  
 3   Monto_Inicial         47860 non-null  int64  
 4   Scoring_Crediticio    47860 non-null  int64  
 5   Meses_Empleo          47860 non-null  int64  
 6   Num_Creditos          47860 non-null  int64  
 7   Ratio_Interes         47860 non-null  float64
 8   Duracion              47860 non-null  int64  
 9   Ratio_Deuda_Ingresos  47860 non-null  float64
 10  Estudios              47860 non-null  object 
 11  Tipo_Jornada_Laboral  47860 non-null  object 
 12  Estado_Civil          47860 non-null  object 
 13  Posesion_Hipoteca     47860 non-null  int64  
 14  Personas_Cargo        47860 non-null  int64  
 15  Proposito             4

In [18]:
condiciones_invalidas = (
    ((df["Edad"] < 19) & (df["Estudios"] == "grado")) |
    ((df["Edad"] < 20) & (df["Estudios"] == "máster")) |
    ((df["Edad"] < 25) & (df["Estudios"] == "doctorado"))
)

df_estudios_invalidos = df[condiciones_invalidas]
len(df_estudios_invalidos)

0

In [19]:
df.shape

(47860, 19)

In [20]:
df.to_excel(
    "C:/Users/ander/OneDrive/Desktop/Reto_07_morado/Datos/Limpios/información_préstamos_limpio.xlsx",index=False)
